# 00 · Environment check

Run each cell top to bottom (Shift+Enter). Every cell should finish without errors.

**Kernel:** in VS Code choose *Select Kernel → Existing Jupyter Server* → `http://localhost:8888` → token `datalake123`.

## 1. Is Spark working?

In [ ]:
import pyspark
print('PySpark version:', pyspark.__version__)

## 2. Can this container see the dataset?

If the list is empty, the CSVs are not in `data/raw/olist/` on your Mac. Run `bash scripts/check_data.sh` in your terminal.

In [ ]:
import os
folder = '/home/jovyan/data/raw/olist'
files = sorted(f for f in os.listdir(folder) if f.endswith('.csv')) if os.path.isdir(folder) else []
print(len(files), 'CSV files found')
for f in files:
    print(' ', f)

## 3. Can Spark read it?

This is the same data we'll stream through Kafka, read here as a plain batch file.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('env-check').master('local[*]').getOrCreate()
orders = spark.read.csv(f'{folder}/olist_orders_dataset.csv', header=True, inferSchema=True)
print('rows:', orders.count())
orders.printSchema()
orders.show(5, truncate=False)

## 4. Can we reach Kafka?

Inside Docker the broker is at `kafka:9092`. This lists the topics that exist (an `orders` topic appears after you run the producer).

In [ ]:
from kafka.admin import KafkaAdminClient

admin = KafkaAdminClient(bootstrap_servers='kafka:9092')
print('Topics:', admin.list_topics())
admin.close()

## All four cells green? Phase 1 is done.

Next: run the producer (README step 6).